# SHRED on ME4OH full-field OFAM3 SST data

Experiment 1: 5 randomly located sensors; eval notebook: Exp1-2-SHRED_OFAM3_SST_static.ipynb

Experiment 2: 1059 randomly located sensors; eval notebook: Exp1-2-SHRED_OFAM3_SST_static.ipynb
- average number of sensors per week in ME4OH SST dataset is 1059 (increases over time)

Experiment 3: 2 week sequences for 5 and 1059 sensors; eval notebook: Exp3-SHRED_OFAM3_SST_static.ipynb

Note: Assumptions as of 30/06/26
- OFAM3 data is stored for Thursdays across the 1979-2014 time period.
- Pending confirmation from the ME4OH team, I have assumed that the data is the full-field SST values generated by the simulator model (OFAM3) averaged across a calendar week

In [ ]:
# Standard imports
import json
import os

# Third-party imports
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler

# Local imports
import models  # https://github.com/Jan-Williams/pyshred/blob/main/models.py 
from Data.preprocess_sst_ofam3_shred import (
    convert_to_anomaly,
    create_shred_sequences,
    load_files,
    split_ordered_data,
)
from helper_functions.plotting_functions import plot_sst_map
from helper_functions.saving import JsonEncoder

np.random.seed(42)

## SHRED preprocessing

In [ ]:
# Experiment 1: 5 randomly located sensors
# exp_index = 1
# num_sensors = 5
# lags = 52
# hidden_dim = 64

# Experiment 2: average number of sensors per week in ME4OH SST dataset is 1059 (increases over time)
# exp_index = 2
# num_sensors = 1059
# lags = 52
# hidden_dim = 64

# Experiment 3: two week sequences
# exp_index = 3
# num_sensors = 1059
# lags = 2
# hidden_dim = 64

# # Experiment 3.1: 1059 sensors, with 250 hidden LSTM dimension, 2 and 52 week sequences
# exp_index = "3-1"
# num_sensors = 1059
# lags = 52
# hidden_dim = 250

# # Experiment 5.1: 1059 sensors, with 250 hidden GRU dimension, 52 week sequences
# # Experiment 5.2: 5 sensors, with 250 hidden GRU dimension, 52 week sequences
# exp_index = 5
# num_sensors = 5
# lags = 52
# hidden_dim = 250
# sequence_model = "GRU"

# Experiment 6: 1059 sensors, 52 week sequences, GRU SHRED (300 hidden dim), SST anomalies
exp_index = 6
num_sensors = 1059
lags = 52
hidden_dim = 300
sequence_model = "GRU"
anomaly = True

In [ ]:
# Load and reformat data
data, dates, lats, longs = load_files()

In [ ]:
print("Preprocessing data...")
train_data, val_data, test_data, train_dates, _, test_dates = split_ordered_data(data, dates, all_dates=True)
if anomaly:
    train_anomaly, val_anomaly, test_anomaly, bulk_mean = convert_to_anomaly((train_data, val_data, test_data))

In [ ]:
# Plots for santity checks:
plot_index = 0
sst_df = pd.DataFrame({
    "Longitude": longs,
    "Latitude": lats,
    "SST": train_data[plot_index]
})
plot_sst_map(sst_df, area="NA", cmap="plasma", title=f"Actual SST, Date: {train_dates[plot_index]}")

anomaly_df = pd.DataFrame({
    "Longitude": longs,
    "Latitude": lats,
    "SST": train_anomaly[plot_index]
})
plot_sst_map(anomaly_df, area="NA", cmap="viridis", title=f"SST Anomaly, Date: {train_dates[plot_index]}")

bulk_df = pd.DataFrame({
    "Longitude": longs,
    "Latitude": lats,
    "SST": bulk_mean
})
plot_sst_map(bulk_df, area="NA", cmap="plasma", title=f"Bulk mean SST, Dates: {train_dates[0]} to {train_dates[-1]}")

In [ ]:
# Normalise data
if anomaly:
    sc = MinMaxScaler()
    sc = sc.fit(train_anomaly)  # Computes min and max from training data only (prevent data leakage)
    train_transformed = sc.transform(train_anomaly)  
    val_transformed = sc.transform(val_anomaly)
    test_transformed = sc.transform(test_anomaly)
else:
    sc = MinMaxScaler()
    sc = sc.fit(train_data)  # Computes min and max from training data only (prevent data leakage)
    train_transformed = sc.transform(train_data)  
    val_transformed = sc.transform(val_data)
    test_transformed = sc.transform(test_data)

# Build sequences
sensor_locations = np.random.choice(data.shape[1], size=num_sensors, replace=False)
sensor_coords=(longs[sensor_locations], lats[sensor_locations])
# print(f"Sensor location indexes will be: {sensor_locations}")

# Build sequences
train_dataset = create_shred_sequences(train_transformed, sensor_locations, lags=lags)
val_dataset = create_shred_sequences(val_transformed, sensor_locations, lags=lags)
test_dataset = create_shred_sequences(test_transformed, sensor_locations, lags=lags)

# Get dates corresponding to the predictions made by the model
test_lag_dates = test_dates[np.array(range(len(test_dates)-lags)) + lags - 1]

## Train + test SHRED model

Experiment 1-3:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
shred = models.SHRED(num_sensors, len(lats), hidden_size=64, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)
validation_errors = models.fit(shred, train_dataset, val_dataset, batch_size=64,
                               num_epochs=1000, lr=1e-3, verbose=True, patience=5)

Exp 5:
shred = models.SHRED(num_sensors, len(lats), seq_model=sequence_model,
                     hidden_size=hidden_dim, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
shred = models.SHRED(num_sensors, len(lats), seq_model=sequence_model,
                     hidden_size=hidden_dim, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)
validation_errors = models.fit(shred, train_dataset, val_dataset, batch_size=64,
                               num_epochs=1000, lr=1e-3, verbose=True, patience=5)

In [ ]:
test_recons = sc.inverse_transform(shred(test_dataset.X).detach().cpu().numpy())
test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())
print('Test Reconstruction Error: ')
print(np.linalg.norm(test_recons - test_ground_truth) / np.linalg.norm(test_ground_truth))

## Save results

In [ ]:
# Make experiment folder
directory = f"Reconstructions/Static/Exp{exp_index}/"
os.makedirs(directory, exist_ok = True)

In [ ]:
np.save(f"Reconstructions/Static/Exp{exp_index}/recons_n{num_sensors}_l{lags}", test_recons)
np.save(f"Reconstructions/Static/Exp{exp_index}/truth_n{num_sensors}_l{lags}", test_ground_truth)

In [ ]:
metadata = {
    "sensor_locations": list(sensor_locations),
    "sensor_coords": list(sensor_coords),
    "test_lag_dates": list(test_lag_dates)
    }

In [ ]:
with open(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}_l{lags}.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, cls=JsonEncoder)